In [ ]:
#  Copyright 2023 Google LLC
#
#  Licensed under the Apache License, Version 2.0 (the "License");
#  you may not use this file except in compliance with the License.
#  You may obtain a copy of the License at
#
#      https://www.apache.org/licenses/LICENSE-2.0
#
#  Unless required by applicable law or agreed to in writing, software
#  distributed under the License is distributed on an "AS IS" BASIS,
#  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#  See the License for the specific language governing permissions and
#  limitations under the License.

# Apply to Lth Target

In [ ]:
from collections import Counter
from IPython.display import SVG, display
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.compiler._component_trace.resource_model import estimate_component_resources
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.text_drawer import draw_text
from flagquantum.dynamic import DynamicCircuit

## `ApplyGateToLthQubit`
A controlled SELECT operation for single-qubit gates.

$$
\mathrm{SELECT} = \sum_{l}|l \rangle \langle l| \otimes [G(l)]_l
$$

Where $G$ is a function that maps an index to a single-qubit gate.

This gate uses the unary iteration scheme to apply `nth_gate(selection)` to the
`selection`-th qubit of `target` all controlled by the `control` register.

#### Parameters
 - `selection_regs`: Indexing `select` signature of type Tuple[`Register`, ...]. It also contains information about the iteration length of each selection register.
 - `nth_gate`: A function mapping the composite selection index to a single-qubit gate.
 - `control_regs`: Control signature for constructing a controlled version of the gate. 

#### References
 - [Encoding Electronic Spectra in Quantum Circuits with Linear T Complexity](https://arxiv.org/abs/1805.03662). Babbush et al. (2018). Section III.A. and Figure 7.


In [ ]:
from examples.algorithms.target_position_catalog import (
    apply_z_to_odd_call_graph, apply_z_to_odd_example,
)

### Example Instances

In [ ]:
op = apply_z_to_odd_example()

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("Apply Z to odd targets", op.signature, {})])))
print("Wire symbols:", " ".join(op.wire_symbols))

## Decomposition

In [ ]:
circuit = DynamicCircuit(op.qubit_count)
circuit.append(op)
ir = circuit.to_ir()
print(draw_text(ir, show_all_qubits=True))
print("Emitted primitive instructions:", dict(Counter(gate.name for gate in ir.instructions)))

### Call Graph

In [ ]:
graph = apply_z_to_odd_call_graph()
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
display(graph.leaf_counts)
resources = estimate_component_resources(op.resource_model())
print("Source model:", resources)
print("Emitted T/T†:", sum(gate.name in ("t", "tdg") for gate in ir.instructions))
print("Reserved wires:", op.qubit_count, "source modeled peak:", resources.peak_logical_qubits)